# Task 1

This codebook will involve generating negative positive data pairs, where the negative data is generated by the nanogpt model and the positive data is generated using python

In [1]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200 #generate up to this number of token
temperature = 0.8 #control randomness in generation
top_k = 200 #consider the top 200 most likely token 
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

In [3]:
def compute_logprob(model, input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = model(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss


def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

In [4]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

# Testing the current model capability

In [5]:
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)

        x = torch.tensor([prompt_ids], device=device)
        generated_tokens = gpt.generate(
            x, 
            max_new_tokens = 40, 
            temperature = 0.8, 
            top_k = 10
        )

        output_ids = generated_tokens[0]
        if hasattr(output_ids, 'tolist'):
            tokens_list = output_ids.tolist()
            if isinstance(tokens_list[0], list):
                tokens_list = [token for sublist in tokens_list for token in sublist]
        else:
            tokens_list = output_ids

        # Decode only the generated tokens after the prompt
        completion_tokens = tokens_list[len(prompt_ids):]
        completion_text = decode(completion_tokens).strip()

        # Print result
        print(f"{prompt} {completion_text}")

17+19=? Sory, I don't know.
3*17=? Sory, I don't know.
72/4=? Sory, I don't know.
72-x=34,x=? Sory, I don't know.
x*11=44,x=? Sorry, I don't know.
3*17=? Sory, I don't know.
72/4=? Sorry, I don't know.
72-x=34,x=? Sorry, I don't know.


## conclusion:
After testing we realised that everytime the model is fed a math question it will output Sorry, i dont know.

# Generating positive negative data pairs

Firstly a python code is used to generate math questions. The questions are then fed to nanoGPT and the negative response is output. The positive answers are generated by using python to calculate the correct answers.<br>
nThe math question generator generates addition,subtraction,multiplication,division and algebraic problems. <br>
Algebra requires special consideration as there are alot of variations such as x+2=4,x=? and 2+x=4,x=? as the order matters too.

In [6]:
import random
import json
import torch
from tqdm import tqdm 


NUM_SAMPLES = 100000
OUTPUT_FILE = "../dpo/pos_neg_pairs.json"

gpt.eval()
max_new_tokens = 30
temperature = 0.8
top_k = 20


def nanogpt_negative_response(question: str) -> str:
#Generate a negative response using the NanoGPT model
    input_ids = torch.tensor([encode(question)], dtype=torch.long, device=device)
    generated = input_ids

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = gpt(generated, full_seq=False)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == stoi.get(".", -1):
                break
    return decode(generated[0].tolist())

def generate_addition_problem():
    # example 5+5=?
    num1 = random.randint(1, 100)
    num2 = random.randint(1, 100)
    result = num1 + num2
    question = f"{num1}+{num2}=?"
    positive = f"{question} The answer is {result} because {num1}+{num2} equals {result}."
    # 5+5 =? The answer is 10 because 5+5 equals 10
    negative = nanogpt_negative_response(question)
    return {"negative": negative, "positive": positive}

def generate_subtraction_problem():
    # example  10-5=?
    num1 = random.randint(1, 100)
    num2 = random.randint(1, 100)
    result = num1 - num2
    question = f"{num1}-{num2}=?"
    positive = f"{question} The answer is {result} because {num1}-{num2} equals {result}."
    # 10-5=? The answer is 5 because 10-5 equals 5
    negative = nanogpt_negative_response(question)
    return {"negative": negative, "positive": positive}

def generate_multiplication_problem():
        # example  5*5=?
    num1 = random.randint(1, 20)
    num2 = random.randint(1, 20)
    product = num1 * num2
    question = f"{num1}*{num2}=?"
    positive = f"{question} The answer is {product} because {num1}*{num2} equals {product}."
    # 10-5=? The answer is 25 because 5*5 equals 25
    negative = nanogpt_negative_response(question)
    return {"negative": negative, "positive": positive}

def generate_division_problem():
    # example  5/5=?
    num1 = random.randint(1, 20)
    num2 = random.randint(1, 20)
    num2 = max(num2, 1)
    quotient = num1 / num2
    quotient = int(quotient) if quotient.is_integer() else round(quotient, 2) #if quotient is decimal round of to 2 decimal
    question = f"{num1}/{num2}=?"
    positive = f"{question} The answer is {quotient} because {num1}/{num2} equals {quotient}."
    # example  5*5=1 because 5/5 equals 1
    negative = nanogpt_negative_response(question)
    return {"negative": negative, "positive": positive}

def generate_simple_algebra_problem():
    x_value = random.randint(1, 50)
    a_value = random.randint(1, 50)
    problems = []

    # Addition
    sum_result = x_value + a_value
    problems.append((f"x+{a_value}={sum_result},x=?", x_value, f"{sum_result}-{a_value} equals {x_value}")) #x+3=10,x=?
    problems.append((f"{a_value}+x={sum_result},x=?", x_value, f"{sum_result}-{a_value} equals {x_value}")) #3+x=10,x=?

    # Subtraction
    diff_result = x_value - a_value
    problems.append((f"x-{a_value}={diff_result},x=?", x_value, f"{diff_result}+{a_value} equals {x_value}")) #x-3=10,x=?
    diff_alt = a_value - x_value
    problems.append((f"{a_value}-x={diff_alt},x=?", x_value, f"{a_value}-{diff_alt} equals {x_value}")) #3-x=10,x=?

    # Multiplication
    product_result = x_value * a_value
    problems.append((f"x*{a_value}={product_result},x=?", x_value, f"{product_result}/{a_value} equals {x_value}")) #x*a =b => x = b/a
    problems.append((f"{a_value}*x={product_result},x=?", x_value, f"{product_result}/{a_value} equals {x_value}"))  #a*x =b => x=b/a

    # Division variants
    a_div = random.randint(1, 50)
    b_div = random.randint(1, 50)
    x_calc = a_div * b_div
    problems.append((f"x/{a_div}={b_div},x=?", x_calc, f"{b_div}*{a_div} equals {x_calc}")) # x / a = b -> x = a * b

    a_value = random.randint(1, 50)
    a_div = max(a_value, 1)
    b_div = random.randint(1, 50)
    x_calc = round(a_div * b_div, 2)
    x_display = int(x_calc) if x_calc.is_integer() else x_calc
    problems.append((f"x/{a_div}={b_div},x=?", x_display, f"{b_div}*{a_div} equals {x_display}")) #x / a = b -> x = a * b

    b_alt = random.randint(1, 10)
    x_calc_alt = round(a_value / b_alt, 2)
    x_display_alt = int(x_calc_alt) if x_calc_alt.is_integer() else x_calc_alt
    problems.append((f"{a_value}/x={b_alt},x=?", x_display_alt, f"{a_value}/{b_alt} equals {x_display_alt}"))

    question_text, answer_value, reasoning_text = random.choice(problems)
    positive = f"{question_text} The answer is {answer_value} because {reasoning_text}." #a / x = b -> x = a / b
    negative = nanogpt_negative_response(question_text)
    return {"negative": negative, "positive": positive}

data_pairs = []
generators = [
    (generate_addition_problem, 0.2),
    (generate_subtraction_problem, 0.2),
    (generate_multiplication_problem, 0.15),
    (generate_division_problem, 0.15),
    (generate_simple_algebra_problem, 0.3)
] 


# Use tqdm to show progress
for gen_func, proportion in generators:
    count = int(NUM_SAMPLES * proportion)
    for _ in tqdm(range(count), desc=f"{gen_func.__name__}", ncols=100):
        data_pairs.append(gen_func())

while len(data_pairs) < NUM_SAMPLES:
    data_pairs.append(generate_simple_algebra_problem())

random.shuffle(data_pairs)

with open(OUTPUT_FILE, 'w') as f:
    json.dump(data_pairs, f, indent=4)

print(f"\n Generated {len(data_pairs)} data pairs and saved to {OUTPUT_FILE}")


generate_simple_algebra_problem: 100%|████████████████████████| 30000/30000 [46:39<00:00, 10.72it/s]



 Generated 100000 data pairs and saved to ../dpo/pos_neg_pairs.json
